In [1]:
%load_ext autoreload
%autoreload 2

# Tabel verb + ma andmete kogumine


**Ülesande püstitus**
    
Kõik laused, kus esineb tabelis [list_ma.csv](list_ma.csv) olev verb ja verbil on otsene alluv deprel=xcomp, feats sisaldab sup

**Tulemus** 

Tabel veergudega:
1. leitud lause, 
2. milline tabelis olevates verbidest seal esineb (algvormis), 
3. ma-infinitiivi vormis oleva verbi lemma, 
4. keeletase.
   

In [4]:
import pandas as pd
from datetime import datetime
from notebook_context import corpus_reader, LISTS_FOLDER

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")



MA_VERBS_LIST =   "./lists/102.list_ma.csv"
RESULTS_FILE= LISTS_FOLDER / f"results/verb_xcompSup_{date_time}.csv"

In [5]:
%%time

# verbid etteantud nimekirjast
df_verbs = pd.read_csv(MA_VERBS_LIST)
my_verbs = list(df_verbs['lemma'].unique())


CPU times: user 1.84 ms, sys: 1.29 ms, total: 3.13 ms
Wall time: 2.41 ms


In [6]:
my_verbs

['agiteerima',
 'ahvatlema',
 'aitama',
 'ajama',
 'ajendama',
 'allutama  ',
 'ambuma',
 'arvama',
 'asetama',
 'asustama',
 'avatlema',
 'ehmatama',
 'ehtima',
 'ergutama',
 'harjama',
 'harjutama',
 'hurjutama',
 'hõikama',
 'hõõruma',
 'häälestama',
 'igatsema',
 'ihalema',
 'innustama',
 'inspireerima',
 'intrigeerima',
 'istutama',
 'juhtima',
 'julgustama',
 'jätma',
 'kaasama',
 'kallutama',
 'kamandama',
 'kangutama',
 'kannustama',
 'kasvatama',
 'keelitama',
 'kehutama',
 'kiskuma',
 'kitsendama ',
 'klõpsama',
 'klõpsatama',
 'kohustama',
 'koolitama',
 'kupatama',
 'kutsuma',
 'käivitama',
 'käratama',
 'käsutama',
 'kütma',
 'laskma',
 'laulma',
 'lihvima',
 'lohistama',
 'looma',
 'lubama',
 'lõõtsuma',
 'läkitama',
 'lööma',
 'lükkama',
 'lülitama',
 'mahitama',
 'mahutama ',
 'majutama',
 'manitsema',
 'manööverdama',
 'matma',
 'meelestama',
 'meelitama',
 'mobiliseerima',
 'motiveerima',
 'mõjutama',
 'mõtlema',
 'määrama',
 'naelutama',
 'nihutama',
 'nõudma',
 'nüh

In [7]:
%%time

collected_data = []
count = 0
for collection_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB") if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): continue
    
    # xcomp
    xcomp_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="xcomp")
   
    if not len(xcomp_nodes): continue
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        for xcomp in xcomp_nodes:
            if xcomp not in kids:
                continue
            if not graph.nodes[xcomp]["feats"] or "VerbForm" not in graph.nodes[xcomp]["feats"].keys() or not graph.nodes[xcomp]["feats"]["VerbForm"] == 'Sup':
                continue
            
            #graph.draw_graph2(highlight=[verb, xcomp])
            d = {
                'id':  graph.get_metadata('row_id'),
                'sentence':  graph.get_metadata('text'),
                'verb':  graph.nodes[verb]["lemma"],
                'xcomp':  graph.nodes[xcomp]["lemma"],
                'form':  graph.nodes[xcomp]["form"],
                'level':  graph.get_metadata('sent_level'),
                'sub': " ".join(
                            [graph.nodes[n]["form"] for n in sorted([verb] + kids)]
                        ),
            }
            
            collected_data.append(d)


../data/Model2Eesti-keele-kui-teise-keele-kooliõpikute-lausete-korpus-2021.conllu
CPU times: user 4.17 s, sys: 38.3 ms, total: 4.21 s
Wall time: 4.26 s


In [8]:
df = pd.DataFrame.from_dict(collected_data)
df.to_csv(RESULTS_FILE, index=None)
df.head()

,id,sentence,verb,xcomp,form,level,sub
0,50,"Uusi kaaslasi oleks vaja tundma õppida, ent ei...",õppima,tundma,tundma,gümn,kaaslasi tundma õppida
1,110,"Mind pani imestama, et Inglismaal käiakse suve...",panema,imestama,imestama,gümn,Mind pani imestama .
2,172,Mis teid sealses elus imestama pani?,panema,imestama,imestama,gümn,Mis teid elus imestama pani ?
3,186,"Mind pani väga imestama, et prantsuse koolides...",panema,imestama,imestama,gümn,Mind pani imestama .
4,373,Kipute ennast ja oma võimalusi natuke liiga sü...,kippuma,kujutama,kujutama,gümn,Kipute kujutama .
